#**MM/GBSA Energy Components - 5FWJ_holo**

In [18]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

rep_files = {
    "Rep 1": "FINAL_RESULTS_MMGBSA_holo_rep1.dat",
    "Rep 2": "FINAL_RESULTS_MMGBSA_holo_rep2.dat",
    "Rep 3": "FINAL_RESULTS_MMGBSA_holo_rep3.dat",
}

# delta variants: Greek Δ (U+0394) and increment ∆ (U+2206)
DELTA = r"[Δ∆]?"

TERMS = ["VDWAALS", "EEL", "EGB", "ESURF", "GGAS", "GSOLV", "TOTAL"]

def extract_delta_section(lines):
    # Find the header line, then take a window after it
    start_idx = None
    for i, ln in enumerate(lines):
        if "Delta (Complex - Receptor - Ligand)" in ln:
            start_idx = i
            break
    if start_idx is None:
        raise ValueError("Cannot find 'Delta (Complex - Receptor - Ligand)' section.")
    return lines[start_idx : min(start_idx + 300, len(lines))]

def parse_term_from_section(section_lines, term):
    # Match lines like: ∆TOTAL   -7.20   1.87  1.70  0.08  0.07
    pat = re.compile(
        rf"^\s*{DELTA}{re.escape(term)}\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s*$"
    )
    for ln in section_lines:
        m = pat.match(ln)
        if m:
            avg, sd_prop, sd, sem_prop, sem = map(float, m.groups())
            return {"avg": avg, "sd": sd, "sem": sem, "line": ln}
    return None

# Run
rep_rows = []
rep_vals = {t: [] for t in TERMS}

for rep, fn in rep_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    lines = p.read_text(encoding="utf-8", errors="ignore").splitlines()
    section = extract_delta_section(lines)

    row = {"Replica": rep, "file": fn}
    for term in TERMS:
        got = parse_term_from_section(section, term)
        if got is None:
            row[f"{term} (avg)"] = np.nan
            row[f"{term} (SD within)"] = np.nan
        else:
            row[f"{term} (avg)"] = got["avg"]
            row[f"{term} (SD within)"] = got["sd"]
            rep_vals[term].append(got["avg"])
            if term == "TOTAL":
                row["Matched TOTAL line"] = got["line"]
    rep_rows.append(row)

df_rep = pd.DataFrame(rep_rows)
display(df_rep)

# Save per-replica table
df_rep.to_csv("MMGBSA_per_replica_holo_summary.csv", index=False, encoding="utf-8-sig")

# Summary mean ± SD across replicas (kcal/mol)
sum_rows = []
for term in TERMS:
    vals = np.array(rep_vals[term], dtype=float)
    if len(vals) != 3:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": "NA (missing)"})
    else:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": f"{vals.mean():.2f} ± {vals.std(ddof=1):.2f}"})

df_summary = pd.DataFrame(sum_rows)
df_summary.loc[df_summary["Term"] == "TOTAL", "Term"] = "ΔGbind (ΔTOTAL)"
display(df_summary)

# Save across-replica summary table
df_summary.to_csv("MMGBSA_across_replicas_holo_summary.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- MMGBSA_per_replica_holo_summary.csv")
print("- MMGBSA_across_replicas_holo_summary.csv")

,Replica,file,VDWAALS (avg),VDWAALS (SD within),EEL (avg),EEL (SD within),EGB (avg),EGB (SD within),ESURF (avg),ESURF (SD within),GGAS (avg),GGAS (SD within),GSOLV (avg),GSOLV (SD within),TOTAL (avg),TOTAL (SD within),Matched TOTAL line
0,Rep 1,FINAL_RESULTS_MMGBSA_holo_rep1.dat,-12.07,11.75,-8.89,11.30,16.64,17.65,-1.65,1.62,-20.96,22.24,14.99,16.11,-5.97,6.61,ΔTOTAL -5.97 8.40 ...
1,Rep 2,FINAL_RESULTS_MMGBSA_holo_rep2.dat,-22.47,2.80,-7.88,3.53,29.87,4.04,-2.94,0.38,-30.35,5.20,26.93,3.89,-3.42,3.72,ΔTOTAL -3.42 1.63 ...
2,Rep 3,FINAL_RESULTS_MMGBSA_holo_rep3.dat,-2.38,5.92,-1.92,4.71,4.52,10.34,-0.32,0.80,-4.30,10.36,4.20,9.57,-0.10,1.51,ΔTOTAL -0.10 1.76 ...


,Term,Mean ± SD (kcal/mol)
0,VDWAALS,-12.31 ± 10.05
1,EEL,-6.23 ± 3.77
2,EGB,17.01 ± 12.68
3,ESURF,-1.64 ± 1.31
4,GGAS,-18.54 ± 13.19
5,GSOLV,15.37 ± 11.37
6,ΔGbind (ΔTOTAL),-3.16 ± 2.94


Saved:
- MMGBSA_per_replica_holo_summary.csv
- MMGBSA_across_replicas_holo_summary.csv


#**MM/GBSA Energy Components - 5FWJ_apo**

In [19]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

rep_files = {
    "Rep 1": "FINAL_RESULTS_MMGBSA_apo_rep1.dat",
    "Rep 2": "FINAL_RESULTS_MMGBSA_apo_rep2.dat",
    "Rep 3": "FINAL_RESULTS_MMGBSA_apo_rep3.dat",
}

# delta variants: Greek Δ (U+0394) and increment ∆ (U+2206)
DELTA = r"[Δ∆]?"

TERMS = ["VDWAALS", "EEL", "EGB", "ESURF", "GGAS", "GSOLV", "TOTAL"]

def extract_delta_section(lines):
    # Find the header line, then take a window after it
    start_idx = None
    for i, ln in enumerate(lines):
        if "Delta (Complex - Receptor - Ligand)" in ln:
            start_idx = i
            break
    if start_idx is None:
        raise ValueError("Cannot find 'Delta (Complex - Receptor - Ligand)' section.")
    return lines[start_idx : min(start_idx + 300, len(lines))]

def parse_term_from_section(section_lines, term):
    # Match lines like: ∆TOTAL   -7.20   1.87  1.70  0.08  0.07
    pat = re.compile(
        rf"^\s*{DELTA}{re.escape(term)}\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s*$"
    )
    for ln in section_lines:
        m = pat.match(ln)
        if m:
            avg, sd_prop, sd, sem_prop, sem = map(float, m.groups())
            return {"avg": avg, "sd": sd, "sem": sem, "line": ln}
    return None

# Run
rep_rows = []
rep_vals = {t: [] for t in TERMS}

for rep, fn in rep_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    lines = p.read_text(encoding="utf-8", errors="ignore").splitlines()
    section = extract_delta_section(lines)

    row = {"Replica": rep, "file": fn}
    for term in TERMS:
        got = parse_term_from_section(section, term)
        if got is None:
            row[f"{term} (avg)"] = np.nan
            row[f"{term} (SD within)"] = np.nan
        else:
            row[f"{term} (avg)"] = got["avg"]
            row[f"{term} (SD within)"] = got["sd"]
            rep_vals[term].append(got["avg"])
            if term == "TOTAL":
                row["Matched TOTAL line"] = got["line"]
    rep_rows.append(row)

df_rep = pd.DataFrame(rep_rows)
display(df_rep)

# Save per-replica table
df_rep.to_csv("MMGBSA_per_replica_apo_summary.csv", index=False, encoding="utf-8-sig")

# Summary mean ± SD across replicas (kcal/mol)
sum_rows = []
for term in TERMS:
    vals = np.array(rep_vals[term], dtype=float)
    if len(vals) != 3:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": "NA (missing)"})
    else:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": f"{vals.mean():.2f} ± {vals.std(ddof=1):.2f}"})

df_summary = pd.DataFrame(sum_rows)
df_summary.loc[df_summary["Term"] == "TOTAL", "Term"] = "ΔGbind (ΔTOTAL)"
display(df_summary)

# Save across-replica summary table
df_summary.to_csv("MMGBSA_across_replicas_apo_summary.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- MMGBSA_per_replica_apo_summary.csv")
print("- MMGBSA_across_replicas_apo_summary.csv")

,Replica,file,VDWAALS (avg),VDWAALS (SD within),EEL (avg),EEL (SD within),EGB (avg),EGB (SD within),ESURF (avg),ESURF (SD within),GGAS (avg),GGAS (SD within),GSOLV (avg),GSOLV (SD within),TOTAL (avg),TOTAL (SD within),Matched TOTAL line
0,Rep 1,FINAL_RESULTS_MMGBSA_apo_rep1.dat,-0.08,0.75,-0.02,0.44,0.11,0.72,-0.01,0.10,-0.10,0.87,0.10,0.65,-0.00,0.37,ΔTOTAL -0.00 1.97 ...
1,Rep 2,FINAL_RESULTS_MMGBSA_apo_rep2.dat,-8.36,8.13,-0.97,1.80,4.56,4.47,-1.03,0.99,-9.33,9.25,3.52,3.54,-5.80,5.91,ΔTOTAL -5.80 1.47 ...
2,Rep 3,FINAL_RESULTS_MMGBSA_apo_rep3.dat,-7.63,5.39,-1.01,1.86,5.17,3.65,-1.08,0.77,-8.64,6.51,4.09,2.99,-4.55,3.86,ΔTOTAL -4.55 1.56 ...


,Term,Mean ± SD (kcal/mol)
0,VDWAALS,-5.36 ± 4.58
1,EEL,-0.67 ± 0.56
2,EGB,3.28 ± 2.76
3,ESURF,-0.71 ± 0.60
4,GGAS,-6.02 ± 5.14
5,GSOLV,2.57 ± 2.16
6,ΔGbind (ΔTOTAL),-3.45 ± 3.05


Saved:
- MMGBSA_per_replica_apo_summary.csv
- MMGBSA_across_replicas_apo_summary.csv


#**MM/GBSA Energy Components - AlphaFold_PAF**

In [20]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

rep_files = {
    "Rep 1": "FINAL_RESULTS_MMGBSA_PAF_rep1.dat",
    "Rep 2": "FINAL_RESULTS_MMGBSA_PAF_rep2.dat",
    "Rep 3": "FINAL_RESULTS_MMGBSA_PAF_rep3.dat",
}

# delta variants: Greek Δ (U+0394) and increment ∆ (U+2206)
DELTA = r"[Δ∆]?"

TERMS = ["VDWAALS", "EEL", "EGB", "ESURF", "GGAS", "GSOLV", "TOTAL"]

def extract_delta_section(lines):
    # Find the header line, then take a window after it
    start_idx = None
    for i, ln in enumerate(lines):
        if "Delta (Complex - Receptor - Ligand)" in ln:
            start_idx = i
            break
    if start_idx is None:
        raise ValueError("Cannot find 'Delta (Complex - Receptor - Ligand)' section.")
    return lines[start_idx : min(start_idx + 300, len(lines))]

def parse_term_from_section(section_lines, term):
    # Match lines like: ∆TOTAL   -7.20   1.87  1.70  0.08  0.07
    pat = re.compile(
        rf"^\s*{DELTA}{re.escape(term)}\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s*$"
    )
    for ln in section_lines:
        m = pat.match(ln)
        if m:
            avg, sd_prop, sd, sem_prop, sem = map(float, m.groups())
            return {"avg": avg, "sd": sd, "sem": sem, "line": ln}
    return None

# Run
rep_rows = []
rep_vals = {t: [] for t in TERMS}

for rep, fn in rep_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    lines = p.read_text(encoding="utf-8", errors="ignore").splitlines()
    section = extract_delta_section(lines)

    row = {"Replica": rep, "file": fn}
    for term in TERMS:
        got = parse_term_from_section(section, term)
        if got is None:
            row[f"{term} (avg)"] = np.nan
            row[f"{term} (SD within)"] = np.nan
        else:
            row[f"{term} (avg)"] = got["avg"]
            row[f"{term} (SD within)"] = got["sd"]
            rep_vals[term].append(got["avg"])
            if term == "TOTAL":
                row["Matched TOTAL line"] = got["line"]
    rep_rows.append(row)

df_rep = pd.DataFrame(rep_rows)
display(df_rep)

# Save per-replica table
df_rep.to_csv("MMGBSA_per_replica_PAF_summary.csv", index=False, encoding="utf-8-sig")

# Summary mean ± SD across replicas (kcal/mol)
sum_rows = []
for term in TERMS:
    vals = np.array(rep_vals[term], dtype=float)
    if len(vals) != 3:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": "NA (missing)"})
    else:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": f"{vals.mean():.2f} ± {vals.std(ddof=1):.2f}"})

df_summary = pd.DataFrame(sum_rows)
df_summary.loc[df_summary["Term"] == "TOTAL", "Term"] = "ΔGbind (ΔTOTAL)"
display(df_summary)

# Save across-replica summary table
df_summary.to_csv("MMGBSA_across_replicas_PAF_summary.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- MMGBSA_per_replica_PAF_summary.csv")
print("- MMGBSA_across_replicas_PAF_summary.csv")

,Replica,file,VDWAALS (avg),VDWAALS (SD within),EEL (avg),EEL (SD within),EGB (avg),EGB (SD within),ESURF (avg),ESURF (SD within),GGAS (avg),GGAS (SD within),GSOLV (avg),GSOLV (SD within),TOTAL (avg),TOTAL (SD within),Matched TOTAL line
0,Rep 1,FINAL_RESULTS_MMGBSA_PAF_rep1.dat,-21.28,2.32,-18.01,3.15,32.86,3.93,-2.71,0.35,-39.29,3.73,30.16,3.73,-9.13,2.03,ΔTOTAL -9.13 1.43 ...
1,Rep 2,FINAL_RESULTS_MMGBSA_PAF_rep2.dat,-9.91,4.70,-5.66,3.69,14.12,6.12,-1.30,0.65,-15.57,6.56,12.82,5.63,-2.75,2.01,ΔTOTAL -2.75 1.84 ...
2,Rep 3,FINAL_RESULTS_MMGBSA_PAF_rep3.dat,-19.38,7.03,-5.61,4.39,23.19,8.05,-2.67,0.98,-24.99,10.11,20.52,7.18,-4.47,4.48,ΔTOTAL -4.47 1.86 ...


,Term,Mean ± SD (kcal/mol)
0,VDWAALS,-16.86 ± 6.09
1,EEL,-9.76 ± 7.14
2,EGB,23.39 ± 9.37
3,ESURF,-2.23 ± 0.80
4,GGAS,-26.62 ± 11.94
5,GSOLV,21.17 ± 8.69
6,ΔGbind (ΔTOTAL),-5.45 ± 3.30


Saved:
- MMGBSA_per_replica_PAF_summary.csv
- MMGBSA_across_replicas_PAF_summary.csv


#**MM/GBSA Energy Components - AlphaFold_CAF**

In [21]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

rep_files = {
    "Rep 1": "FINAL_RESULTS_MMGBSA_CAF_rep1.dat",
    "Rep 2": "FINAL_RESULTS_MMGBSA_CAF_rep2.dat",
    "Rep 3": "FINAL_RESULTS_MMGBSA_CAF_rep3.dat",
}

# delta variants: Greek Δ (U+0394) and increment ∆ (U+2206)
DELTA = r"[Δ∆]?"

TERMS = ["VDWAALS", "EEL", "EGB", "ESURF", "GGAS", "GSOLV", "TOTAL"]

def extract_delta_section(lines):
    # Find the header line, then take a window after it
    start_idx = None
    for i, ln in enumerate(lines):
        if "Delta (Complex - Receptor - Ligand)" in ln:
            start_idx = i
            break
    if start_idx is None:
        raise ValueError("Cannot find 'Delta (Complex - Receptor - Ligand)' section.")
    return lines[start_idx : min(start_idx + 300, len(lines))]

def parse_term_from_section(section_lines, term):
    # Match lines like: ∆TOTAL   -7.20   1.87  1.70  0.08  0.07
    pat = re.compile(
        rf"^\s*{DELTA}{re.escape(term)}\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s+"
        rf"([-+]?\d+(?:\.\d+)?)\s*$"
    )
    for ln in section_lines:
        m = pat.match(ln)
        if m:
            avg, sd_prop, sd, sem_prop, sem = map(float, m.groups())
            return {"avg": avg, "sd": sd, "sem": sem, "line": ln}
    return None

# Run
rep_rows = []
rep_vals = {t: [] for t in TERMS}

for rep, fn in rep_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    lines = p.read_text(encoding="utf-8", errors="ignore").splitlines()
    section = extract_delta_section(lines)

    row = {"Replica": rep, "file": fn}
    for term in TERMS:
        got = parse_term_from_section(section, term)
        if got is None:
            row[f"{term} (avg)"] = np.nan
            row[f"{term} (SD within)"] = np.nan
        else:
            row[f"{term} (avg)"] = got["avg"]
            row[f"{term} (SD within)"] = got["sd"]
            rep_vals[term].append(got["avg"])
            if term == "TOTAL":
                row["Matched TOTAL line"] = got["line"]
    rep_rows.append(row)

df_rep = pd.DataFrame(rep_rows)
display(df_rep)

# Save per-replica table
df_rep.to_csv("MMGBSA_per_replica_CAF_summary.csv", index=False, encoding="utf-8-sig")

# Summary mean ± SD across replicas (kcal/mol)
sum_rows = []
for term in TERMS:
    vals = np.array(rep_vals[term], dtype=float)
    if len(vals) != 3:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": "NA (missing)"})
    else:
        sum_rows.append({"Term": term, "Mean ± SD (kcal/mol)": f"{vals.mean():.2f} ± {vals.std(ddof=1):.2f}"})

df_summary = pd.DataFrame(sum_rows)
df_summary.loc[df_summary["Term"] == "TOTAL", "Term"] = "ΔGbind (ΔTOTAL)"
display(df_summary)

# Save across-replica summary table
df_summary.to_csv("MMGBSA_across_replicas_CAF_summary.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- MMGBSA_per_replica_CAF_summary.csv")
print("- MMGBSA_across_replicas_CAF_summary.csv")

,Replica,file,VDWAALS (avg),VDWAALS (SD within),EEL (avg),EEL (SD within),EGB (avg),EGB (SD within),ESURF (avg),ESURF (SD within),GGAS (avg),GGAS (SD within),GSOLV (avg),GSOLV (SD within),TOTAL (avg),TOTAL (SD within),Matched TOTAL line
0,Rep 1,FINAL_RESULTS_MMGBSA_CAF_rep1.dat,-17.59,2.33,-8.89,2.45,21.90,2.16,-2.62,0.24,-26.48,2.84,19.28,2.05,-7.20,1.70,ΔTOTAL -7.20 1.87 ...
1,Rep 2,FINAL_RESULTS_MMGBSA_CAF_rep2.dat,-14.16,7.12,-8.76,7.78,19.32,10.10,-2.05,0.99,-22.92,14.16,17.27,9.15,-5.64,5.28,ΔTOTAL -5.64 2.43 ...
2,Rep 3,FINAL_RESULTS_MMGBSA_CAF_rep3.dat,-10.43,5.80,-1.29,7.50,9.21,9.68,-1.42,0.89,-11.72,12.53,7.80,8.87,-3.93,4.10,ΔTOTAL -3.93 5.13 ...


,Term,Mean ± SD (kcal/mol)
0,VDWAALS,-14.06 ± 3.58
1,EEL,-6.31 ± 4.35
2,EGB,16.81 ± 6.71
3,ESURF,-2.03 ± 0.60
4,GGAS,-20.37 ± 7.70
5,GSOLV,14.78 ± 6.13
6,ΔGbind (ΔTOTAL),-5.59 ± 1.64


Saved:
- MMGBSA_per_replica_CAF_summary.csv
- MMGBSA_across_replicas_CAF_summary.csv


#**MM/GBSA Summary Across All Systems**

In [22]:
import pandas as pd
from pathlib import Path

# ---------------------------
# Input files
# ---------------------------
per_replica_files = {
    "holo": "MMGBSA_per_replica_holo_summary.csv",
    "apo": "MMGBSA_per_replica_apo_summary.csv",
    "PAF": "MMGBSA_per_replica_PAF_summary.csv",
    "CAF": "MMGBSA_per_replica_CAF_summary.csv",
}

across_replica_files = {
    "holo": "MMGBSA_across_replicas_holo_summary.csv",
    "apo": "MMGBSA_across_replicas_apo_summary.csv",
    "PAF": "MMGBSA_across_replicas_PAF_summary.csv",
    "CAF": "MMGBSA_across_replicas_CAF_summary.csv",
}

# ---------------------------
# Combine per-replica tables
# ---------------------------
per_replica_dfs = []

for system, fn in per_replica_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    df = pd.read_csv(p)
    df.insert(0, "System", system)   # add System as first column
    per_replica_dfs.append(df)

df_per_replica_all = pd.concat(per_replica_dfs, ignore_index=True)
display(df_per_replica_all)

df_per_replica_all.to_csv(
    "MMGBSA_per_replica_all_systems.csv",
    index=False,
    encoding="utf-8-sig"
)

# ---------------------------
# Combine across-replica tables
# ---------------------------
across_replica_dfs = []

for system, fn in across_replica_files.items():
    p = Path(fn)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {fn}")

    df = pd.read_csv(p)
    df.insert(0, "System", system)   # add System as first column
    across_replica_dfs.append(df)

df_across_replica_all = pd.concat(across_replica_dfs, ignore_index=True)
display(df_across_replica_all)

df_across_replica_all.to_csv(
    "MMGBSA_across_replicas_all_systems.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print("- MMGBSA_per_replica_all_systems.csv")
print("- MMGBSA_across_replicas_all_systems.csv")

,System,Replica,file,VDWAALS (avg),VDWAALS (SD within),EEL (avg),EEL (SD within),EGB (avg),EGB (SD within),ESURF (avg),ESURF (SD within),GGAS (avg),GGAS (SD within),GSOLV (avg),GSOLV (SD within),TOTAL (avg),TOTAL (SD within),Matched TOTAL line
0,holo,Rep 1,FINAL_RESULTS_MMGBSA_holo_rep1.dat,-12.07,11.75,-8.89,11.30,16.64,17.65,-1.65,1.62,-20.96,22.24,14.99,16.11,-5.97,6.61,ΔTOTAL -5.97 8.40 ...
1,holo,Rep 2,FINAL_RESULTS_MMGBSA_holo_rep2.dat,-22.47,2.80,-7.88,3.53,29.87,4.04,-2.94,0.38,-30.35,5.20,26.93,3.89,-3.42,3.72,ΔTOTAL -3.42 1.63 ...
2,holo,Rep 3,FINAL_RESULTS_MMGBSA_holo_rep3.dat,-2.38,5.92,-1.92,4.71,4.52,10.34,-0.32,0.80,-4.30,10.36,4.20,9.57,-0.10,1.51,ΔTOTAL -0.10 1.76 ...
3,apo,Rep 1,FINAL_RESULTS_MMGBSA_apo_rep1.dat,-0.08,0.75,-0.02,0.44,0.11,0.72,-0.01,0.10,-0.10,0.87,0.10,0.65,-0.00,0.37,ΔTOTAL -0.00 1.97 ...
4,apo,Rep 2,FINAL_RESULTS_MMGBSA_apo_rep2.dat,-8.36,8.13,-0.97,1.80,4.56,4.47,-1.03,0.99,-9.33,9.25,3.52,3.54,-5.80,5.91,ΔTOTAL -5.80 1.47 ...
5,apo,Rep 3,FINAL_RESULTS_MMGBSA_apo_rep3.dat,-7.63,5.39,-1.01,1.86,5.17,3.65,-1.08,0.77,-8.64,6.51,4.09,2.99,-4.55,3.86,ΔTOTAL -4.55 1.56 ...
6,PAF,Rep 1,FINAL_RESULTS_MMGBSA_PAF_rep1.dat,-21.28,2.32,-18.01,3.15,32.86,3.93,-2.71,0.35,-39.29,3.73,30.16,3.73,-9.13,2.03,ΔTOTAL -9.13 1.43 ...
7,PAF,Rep 2,FINAL_RESULTS_MMGBSA_PAF_rep2.dat,-9.91,4.70,-5.66,3.69,14.12,6.12,-1.30,0.65,-15.57,6.56,12.82,5.63,-2.75,2.01,ΔTOTAL -2.75 1.84 ...
8,PAF,Rep 3,FINAL_RESULTS_MMGBSA_PAF_rep3.dat,-19.38,7.03,-5.61,4.39,23.19,8.05,-2.67,0.98,-24.99,10.11,20.52,7.18,-4.47,4.48,ΔTOTAL -4.47 1.86 ...
9,CAF,Rep 1,FINAL_RESULTS_MMGBSA_CAF_rep1.dat,-17.59,2.33,-8.89,2.45,21.90,2.16,-2.62,0.24,-26.48,2.84,19.28,2.05,-7.20,1.70,ΔTOTAL -7.20 1.87 ...


,System,Term,Mean ± SD (kcal/mol)
0,holo,VDWAALS,-12.31 ± 10.05
1,holo,EEL,-6.23 ± 3.77
2,holo,EGB,17.01 ± 12.68
3,holo,ESURF,-1.64 ± 1.31
4,holo,GGAS,-18.54 ± 13.19
5,holo,GSOLV,15.37 ± 11.37
6,holo,ΔGbind (ΔTOTAL),-3.16 ± 2.94
7,apo,VDWAALS,-5.36 ± 4.58
8,apo,EEL,-0.67 ± 0.56
9,apo,EGB,3.28 ± 2.76


Saved:
- MMGBSA_per_replica_all_systems.csv
- MMGBSA_across_replicas_all_systems.csv
